# LLM Memory

LLMs are stateless — every call is independent. To support multi-turn conversations we must store history ourselves and inject it into the prompt. This notebook shows two LangChain memory strategies.

In [ ]:
%pip install -q openai langchain langchain_openai langchain_core langchain_community python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in a .env file"
print("API key loaded.")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

query_chain = llm | StrOutputParser()

Without memory each call is independent — the model forgets everything between calls.

In [ ]:
query_chain.invoke("Hello, my name is Pavel")

In [ ]:
query_chain.invoke("What is my name?")  # will not know

## ConversationBufferWindowMemory

Keeps the last `k` conversation turns verbatim. Older turns are dropped.

In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(k=10)
conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

In [ ]:
response = conversation.invoke("Hello, my name is Pavel")
print(response["response"])

In [ ]:
response = conversation.invoke("What is my name?")
print(response["response"])  # now it knows

In [ ]:
# Inspect raw stored history
print(conversation.memory.load_memory_variables({}))

## ConversationSummaryMemory

Uses the LLM to summarise past turns instead of storing them verbatim. Handles long conversations without token overflow.

In [ ]:
from langchain.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(llm=llm)
conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

In [ ]:
response = conversation.invoke("Hello, my name is Pavel")
print(response["response"])

In [ ]:
response = conversation.invoke("Oh sorry, I was wrong — my name is Bob")
print(response["response"])

In [ ]:
response = conversation.invoke("What is my name?")
print(response["response"])

In [ ]:
# The stored memory is an LLM-generated summary, not the raw transcript
print(conversation.memory.load_memory_variables({}))

## Pre-loading memory from history

In [ ]:
from langchain.memory import ChatMessageHistory

history = ChatMessageHistory()
history.add_user_message("My name is Pavel")
history.add_user_message("My favourite color is blue")

memory = ConversationSummaryMemory.from_messages(llm=llm, chat_memory=history)
conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

In [ ]:
response = conversation.invoke("What is my name and what colour is the best?")
print(response["response"])

## Other memory types

- **ConversationBufferMemory** — stores entire history, no cap (risk of overflow).
- **ConversationSummaryBufferMemory** — keeps recent turns verbatim + summarises older ones. Best of both worlds.
- **ConversationTokenBufferMemory** — flushes based on token count rather than number of turns.